# nanochat d12 → SFT → WebUI: Gutenberg Edition (pre-tokenized)

Trains a d12 base model on Project Gutenberg English books.

**What changed vs the old gutenbergv2:** the corpus is now **pre-tokenized once** into flat `uint16` token shards (Cell 5b) and training reads those directly. nanochat's default loader tokenizes documents *inside* the training loop, which is fine for short ClimbMix web docs but stalls the GPU on Gutenberg's book-length documents — dropping MFU and pushing the d12 ETA well past 2h. Reading pre-tokenized tokens keeps the GPU saturated, so the run lands back around **~110–120 min** at ~56% MFU, matching the `shit_gpt` ClimbMix run.

## Cell 1 — Clone + install deps (skip torch, keep Colab's preinstalled CUDA build)

In [1]:
import os
if not os.path.isdir('/content/nanochat'):
    !git clone https://github.com/karpathy/nanochat
%cd /content/nanochat
!pip install -q rustbpe tiktoken tokenizers datasets wandb fastapi uvicorn psutil kernels

Cloning into 'nanochat'...
remote: Enumerating objects: 1783, done.
remote: Total 1783 (delta 0), reused 0 (delta 0), pack-reused 1783 (from 1)
Receiving objects: 100% (1783/1783), 1.91 MiB | 30.60 MiB/s, done.
Resolving deltas: 100% (1135/1135), done.
/content/nanochat
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.1 MB/s eta 0:00:00


## Cell 2 — Mount Drive + symlink checkpoints/tokenizer (re-run safe)

Data stays on fast local SSD (Drive FUSE chokes on parquet streaming). Tokenizer + all checkpoint dirs symlink to Drive so they survive session restarts.

In [2]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted, skipping")

LOCAL = '/content/nanochat_cache'
DRIVE = '/content/drive/MyDrive/Think.Genesis'

# base_data_gutenberg_tok holds the pre-tokenized uint16 shards; symlinking it to
# Drive means we tokenize once and reuse across session restarts.
SUBDIRS = ['tokenizer', 'base_checkpoints', 'chatsft_checkpoints',
           'chatrl_checkpoints', 'base_data_gutenberg_tok']

os.makedirs(LOCAL, exist_ok=True)
for sub in SUBDIRS:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

def ensure_symlink(link_path, target):
    if os.path.islink(link_path):
        if os.readlink(link_path) == target:
            return
        os.unlink(link_path)
    elif os.path.exists(link_path):
        raise RuntimeError(f"{link_path} exists and is not a symlink; remove manually")
    os.symlink(target, link_path)

for sub in SUBDIRS:
    ensure_symlink(f'{LOCAL}/{sub}', f'{DRIVE}/{sub}')

os.environ['NANOCHAT_BASE_DIR'] = LOCAL
print("Base dir:", LOCAL)
for sub in SUBDIRS:
    print(f"  {sub:25s} -> {os.readlink(f'{LOCAL}/{sub}')}")

Mounted at /content/drive
Base dir: /content/nanochat_cache
  tokenizer                 -> /content/drive/MyDrive/Think.Genesis/tokenizer
  base_checkpoints          -> /content/drive/MyDrive/Think.Genesis/base_checkpoints
  chatsft_checkpoints       -> /content/drive/MyDrive/Think.Genesis/chatsft_checkpoints
  chatrl_checkpoints        -> /content/drive/MyDrive/Think.Genesis/chatrl_checkpoints
  base_data_gutenberg_tok   -> /content/drive/MyDrive/Think.Genesis/base_data_gutenberg_tok


## Cell 2b — Install our pre-tokenization code into the fresh clone (run every session)

The three files we authored (`pretok_gutenberg.py`, `pretok_dataloader.py`, and our
modified `base_train.py`) live in the **project root on Drive** (`DRIVE`), not inside
the disposable `nanochat` clone. This cell copies them into the clone so they can be
invoked as `python -m scripts.pretok_gutenberg` / `--pretokenized`. Re-run every
session after the clone in Cell 1. Edit the originals at the project root.

In [3]:
import os, shutil

# DRIVE (defined in Cell 2) is the project root that holds our authored files.
REPO = '/content/nanochat'
FILES = [
    # (source at project root,        destination inside the clone)
    ('pretok_gutenberg.py',  f'{REPO}/scripts/pretok_gutenberg.py'),
    ('base_train.py',        f'{REPO}/scripts/base_train.py'),       # our modified version
    ('pretok_dataloader.py', f'{REPO}/nanochat/pretok_dataloader.py'),
]

for rel, dst in FILES:
    srcp = os.path.join(DRIVE, rel)
    assert os.path.exists(srcp), f"Missing source file at project root: {srcp}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(srcp, dst)
    print(f"installed {rel:24s} -> {dst}")

print("\nOur pre-tokenization code is now in the clone; scripts.* imports will resolve.")

installed pretok_gutenberg.py      -> /content/nanochat/scripts/pretok_gutenberg.py
installed base_train.py            -> /content/nanochat/scripts/base_train.py
installed pretok_dataloader.py     -> /content/nanochat/nanochat/pretok_dataloader.py

Our pre-tokenization code is now in the clone; scripts.* imports will resolve.


## Cell 3 — Sanity check GPU + imports

In [4]:
import torch, rustbpe, tiktoken
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
print(f"gpu: {torch.cuda.get_device_name(0)}")
print(f"capability: sm{''.join(map(str, torch.cuda.get_device_capability(0)))}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-40GB
capability: sm80
bf16 supported: True


Want to see `NVIDIA A100-SXM4-40GB`, `sm80`, `bf16 supported: True`. If not A100, stop and re-request a GPU runtime.

## Cell 4 — Download all 100 Gutenberg shards to Drive (one-time, ~20–40 min)

Downloads `sedthh/gutenberg_english` from HuggingFace, splits into 100 ~107MB parquet shards,
and writes them to Drive for permanent storage. Re-run safe — skips shards that already exist.

In [5]:
import os, json, time, sys
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset

NUM_SHARDS = 100
GUTENBERG_HF_ID = "sedthh/gutenberg_english"
DRIVE = '/content/drive/MyDrive/Think.Genesis'
drive_dir = os.path.join(DRIVE, 'gutenberg_shards')
os.makedirs(drive_dir, exist_ok=True)

progress_path = os.path.join(drive_dir, '_progress.json')
if os.path.exists(progress_path):
    p = json.load(open(progress_path))
    if p.get('complete'):
        print(f"Already complete: {p['total_books']:,} books across {NUM_SHARDS} shards in {drive_dir}")
else:
    print("Loading dataset index from HuggingFace (may take a minute)...")
    sys.stdout.flush()
    ds = load_dataset(GUTENBERG_HF_ID, split='train')
    total = len(ds)
    print(f"Books: {total:,}")
    sys.stdout.flush()

    rows_per_shard = total // NUM_SHARDS
    t0 = time.time()

    for shard_idx in range(NUM_SHARDS):
        shard_path = os.path.join(drive_dir, f'shard_{shard_idx:05d}.parquet')
        if os.path.exists(shard_path):
            print(f"  shard_{shard_idx:05d}.parquet already exists, skipping.")
            continue

        start = shard_idx * rows_per_shard
        end   = total if shard_idx == NUM_SHARDS - 1 else start + rows_per_shard
        shard_ds = ds.select(range(start, end))

        texts = []
        for i, row in enumerate(shard_ds):
            text = (row.get('TEXT') or '').strip()
            if text:
                texts.append(text)
            if (i + 1) % 500 == 0:
                print(f"  shard {shard_idx+1}/{NUM_SHARDS}  row {i+1:,}/{end-start:,}  "
                      f"elapsed={time.time()-t0:.0f}s")

        tmp = shard_path + '.tmp'
        table = pa.table({'text': texts})
        pq.write_table(table, tmp, compression='snappy')
        os.replace(tmp, shard_path)
        print(f"  wrote shard_{shard_idx:05d}.parquet  ({len(texts):,} books)")

    json.dump({'complete': True, 'total_books': total, 'num_shards': NUM_SHARDS},
              open(progress_path, 'w'))
    print(f"\nDone. {total:,} books across {NUM_SHARDS} shards in {time.time()-t0:.0f}s")

print(f"\nShards on Drive: {len([f for f in os.listdir(drive_dir) if f.endswith('.parquet')])}")

Already complete: 48,284 books across 100 shards in /content/drive/MyDrive/Think.Genesis/gutenberg_shards

Shards on Drive: 100


Should print `100`. Run this once and leave Drive alone — Cell 4b handles the per-run shard selection.

## Cell 4b — Stage Gutenberg shards from Drive → local SSD (run every session)

Copies a handful of Gutenberg parquet shards from Drive to the local SSD data dir
(`base_data_climbmix`, nanochat's default source dir name). 8 shards is far more
raw text than the ~1.5B tokens the d12 run needs — the pre-tokenizer in Cell 5b
stops once it has enough. The val split is carved out by the pre-tokenizer, not by
file ordering, so no special "last shard" handling is needed here.

In [6]:
import os, random, shutil, time

STAGE_SHARDS = 8   # ~8 Gutenberg shards is plenty of raw text for ~1.5B tokens
DRIVE        = '/content/drive/MyDrive/Think.Genesis'
drive_dir    = os.path.join(DRIVE, 'gutenberg_shards')
local_dir    = os.path.join(os.environ['NANOCHAT_BASE_DIR'], 'base_data_climbmix')

os.makedirs(local_dir, exist_ok=True)

# Clear any previously staged shards so we only tokenize this run's selection
for f in os.listdir(local_dir):
    if f.endswith('.parquet'):
        os.remove(os.path.join(local_dir, f))

all_shards = sorted([f for f in os.listdir(drive_dir) if f.endswith('.parquet')])
assert len(all_shards) >= STAGE_SHARDS, f"Only {len(all_shards)} shards on Drive, need {STAGE_SHARDS}"
chosen = sorted(random.sample(all_shards, STAGE_SHARDS))

print(f"Staging {STAGE_SHARDS} shards to {local_dir}\n")
t0 = time.time()
for i, fname in enumerate(chosen):
    src = os.path.join(drive_dir, fname)
    dst = os.path.join(local_dir, fname)
    print(f"  [{i+1}/{STAGE_SHARDS}] copying {fname} ...", end=' ', flush=True)
    shutil.copy2(src, dst)
    print(f"{os.path.getsize(dst)/1024/1024:.0f} MB")

print(f"\nDone in {time.time()-t0:.0f}s — {STAGE_SHARDS} shards ready on local SSD")
print("Shards staged:", sorted(os.listdir(local_dir)))

Staging 8 shards to /content/nanochat_cache/base_data_climbmix

  [1/8] copying shard_00003.parquet ... 115 MB
  [2/8] copying shard_00015.parquet ... 101 MB
  [3/8] copying shard_00028.parquet ... 64 MB
  [4/8] copying shard_00029.parquet ... 72 MB
  [5/8] copying shard_00044.parquet ... 102 MB
  [6/8] copying shard_00075.parquet ... 112 MB
  [7/8] copying shard_00089.parquet ... 87 MB
  [8/8] copying shard_00092.parquet ... 84 MB

Done in 82s — 8 shards ready on local SSD
Shards staged: ['shard_00003.parquet', 'shard_00015.parquet', 'shard_00028.parquet', 'shard_00029.parquet', 'shard_00044.parquet', 'shard_00075.parquet', 'shard_00089.parquet', 'shard_00092.parquet']


## Cell 5 — Train tokenizer

The tokenizer is now baked into the pre-tokenized data (Cell 5b), so it's worth
training it properly. The smoke-test line is fine for a quick end-to-end dry run;
use the full `tok_train` (no `--max-chars`) for a real run.

In [7]:
# Smoke test first (~3-5 min):
!python -m scripts.tok_train --max-chars=200000000
# Full version (uncomment for real run, ~20-40 min, no progress bar):
# !python -m scripts.tok_train
!python -m scripts.tok_eval

max_chars: 200,000,000
doc_cap: 10,000
vocab_size: 32,768
2026-05-29 17:20:56,289 - rustbpe - INFO - Processing sequences from iterator (buffer_size: 8192)
2026-05-29 17:21:05,197 - rustbpe - INFO - Processed 3374 sequences total, 219609 unique
2026-05-29 17:21:05,211 - rustbpe - INFO - Starting BPE training: 32503 merges to compute
2026-05-29 17:21:05,211 - rustbpe - INFO - Computing initial pair counts from 219609 unique sequences
2026-05-29 17:21:05,384 - rustbpe - INFO - Building heap with 4988 unique pairs
2026-05-29 17:21:05,385 - rustbpe - INFO - Starting merge loop
2026-05-29 17:21:05,682 - rustbpe - INFO - Progress: 1% (326/32503 merges) - Last merge: (488, 297) -> 581 (frequency: 9228)
2026-05-29 17:21:05,744 - rustbpe - INFO - Progress: 2% (651/32503 merges) - Last merge: (409, 565) -> 906 (frequency: 3669)
2026-05-29 17:21:05,782 - rustbpe - INFO - Progress: 3% (976/32503 merges) - Last merge: (586, 116) -> 1231 (frequency: 2225)
2026-05-29 17:21:05,807 - rustbpe - INFO - P

## Cell 5b — Pre-tokenize Gutenberg → flat uint16 token shards (run once per session)

This is the key change. It tokenizes the staged books **once** into a flat token
stream on disk (`base_data_gutenberg_tok/`, symlinked to Drive), each book prefixed
with `<|bos|>`. Training then memory-maps these shards and reads contiguous `T+1`
windows with **zero tokenization in the loop**, so the GPU stays fed at full MFU.

- `--target-tokens 1_500_000_000` → ~1.5B train tokens, enough for the d12 horizon
  (~1.32B at ratio=12) plus headroom. The val shard is carved off separately.
- Re-run safe: if `meta.json` already marks it complete, it skips instantly.
- Expect a few minutes (tokenization runs at millions of tokens/sec on CPU).

In [8]:
# Pre-tokenize once. Writes uint16 .bin shards + meta.json into
# $NANOCHAT_BASE_DIR/base_data_gutenberg_tok (symlinked to Drive).
!OMP_NUM_THREADS=1 python -m scripts.pretok_gutenberg \
    --target-tokens 1500000000 \
    --val-tokens 20000000

import os, json
meta = json.load(open(os.path.join(os.environ['NANOCHAT_BASE_DIR'],
                                   'base_data_gutenberg_tok', 'meta.json')))
print(meta)

Tokenizer vocab size: 32,768 | BOS id: 32759
Found 8 source parquet shards in /content/nanochat_cache/base_data_climbmix
  books=2,560  tokens=199,463,587  (3.84M tok/s)  elapsed=52s

Done in 78s
  books tokenized : 3,856
  train tokens    : 284,857,175 across 3 shards
  val tokens      : 20,000,000 across 1 shard(s)
  throughput      : 3.89M tok/s
  output dir      : /content/nanochat_cache/base_data_gutenberg_tok
{'complete': True, 'dtype': 'uint16', 'bos_token_id': 32759, 'vocab_size': 32768, 'total_tokens': 304857175, 'train_tokens': 284857175, 'val_tokens': 20000000, 'num_train_shards': 3, 'num_val_shards': 1, 'tokens_per_shard': 100000000, 'num_books': 3856}


## Cell 6 — Pretrain d12 base model (~2h on A100 40GB)

In [9]:
!OMP_NUM_THREADS=1 python -m scripts.base_train \
    --depth=12 \
    --pretokenized \
    --window-pattern=L \
    --device-batch-size=16 \
    --save-every=500 \
    --core-metric-every=-1 \
    --sample-every=-1 \
    --run=dummy


                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
2026-05-29 17:25:38,112 - nanochat.common - INFO - Distributed world size: 1
GPU: NVIDIA A100-SXM4-40GB | Peak FLOPS (BF16): 3.12e+14
COMPUTE_DTYPE: torch.bfloat16 (auto-detected: CUDA SM 80 (bf16 supported))
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!

Key flags:

- `--pretokenized` → read the flat uint16 token shards from Cell 5b (no in-loop
  tokenization). This is what keeps MFU high and the ETA near ~110–120 min.
- `--window-pattern=L` → SDPA dispatches to FA2 (the ~56% MFU path)
- `--save-every=500` → ~5 checkpoints across the run, survive session restarts
- `--core-metric-every=-1`, `--sample-every=-1` → skip in-loop evals (run separately)

You should see `Using pre-tokenized flat dataloader` near the top and per-step
`dt` around ~2.6s at `bf16_mfu` ~56, with `eta` ~110m — same ballpark as the
`shit_gpt` ClimbMix run. If a session dies, find the latest step in Drive and resume:

In [ ]:
import os, glob
ckpts = sorted(glob.glob('/content/drive/MyDrive/Think.Genesis/base_checkpoints/d12/*'))
print("Latest checkpoints:", [os.path.basename(c) for c in ckpts[-3:]])

Then re-run Cell 6 with `--resume-from-step=N` added.

## Cell 7 — Sanity-check the base model (text completion)

In [10]:
!python -m scripts.base_eval --eval sample --model-tag d12 --device-batch-size=16

Autodetected device type: cuda
2026-05-29 19:34:55,760 - nanochat.common - INFO - Distributed world size: 1
2026-05-29 19:34:55,762 - nanochat.checkpoint_manager - INFO - Loading model from /content/nanochat_cache/base_checkpoints/d12 with step 2520
2026-05-29 19:34:57,035 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 12, 'n_head': 6, 'n_kv_head': 6, 'n_embd': 768, 'window_pattern': 'L'}
Evaluating model: base_model (step 2520)
Eval modes: sample

Model Samples

Conditioned samples:
--------------------------------------------------------------------------------
<|bos|>The capital of France is the

  capital of the world, and the capital of the

  world
--------------------------------------------------------------------------------
<|bos|>The chemical symbol of gold is the

  symbol of the chemical symbol of the chemical

  symbol of the
-------------------------------------------------------------------------

You'll see completions like "The capital of France is Paris". If this is total garbage, base training failed — don't continue to SFT.

For arbitrary prompts:

In [11]:
import torch
from nanochat.common import compute_init, autodetect_device_type
from nanochat.engine import Engine
from nanochat.checkpoint_manager import load_model

_, _, _, _, device = compute_init(autodetect_device_type())
model, tokenizer, _ = load_model("base", device, phase="eval", model_tag="d12")
engine = Engine(model, tokenizer)

def complete(prompt, max_tokens=64, temperature=0.8, top_k=50):
    tokens = tokenizer(prompt, prepend="<|bos|>")
    out, _ = engine.generate_batch(tokens, num_samples=1, max_tokens=max_tokens,
                                    temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0])

print(complete("Once upon a time"))
print(complete("Q: What is the capital of France?\nA:"))

Autodetected device type: cuda
<|bos|>Once upon a time there was a young girl who used to love

the man she loved.
But the love that came into the girl’s heart was a different

feeling. To love in the way that made her loving, it did not

make her jealous. The love that came to her was a kind of

jealousy.
<|bos|>Q: What is the capital of France?
A: 1868, 1875, 1876, 1877; 1878, 1879, 1881, 1882, 1883, 1884; 1894, 1895, 1896, 1897, 1898.
Roy
Charles


## Cell 8 — Download SFT identity-conversation data

In [12]:
!curl -L -o $NANOCHAT_BASE_DIR/identity_conversations.jsonl \
    https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl
!ls -lh $NANOCHAT_BASE_DIR/identity_conversations.jsonl

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2452k  100 2452k    0     0  1400k      0  0:00:01  0:00:01 --:--:-- 1400k
-rw-r--r-- 1 root root 2.4M May 29 19:41 /content/nanochat_cache/identity_conversations.jsonl


SmolTalk, MMLU, GSM8K auto-download from HF on first SFT run — no manual fetch needed.

## Cell 9 — SFT the d12 base model (~30–60 min)

In [ ]:
!OMP_NUM_THREADS=1 python -m scripts.chat_sft \
    --model-tag=d12 \
    --device-batch-size=8 \
    --eval-every=-1 \
    --chatcore-every=-1 \
    --run=dummy

- `--device-batch-size=8` → conservative for SFT; bump to 16 if no OOM
- `--eval-every=-1 --chatcore-every=-1` → skip slow single-GPU evals during training

Output: `chatsft_checkpoints/d12/` (auto-synced to Drive via the symlink).

## Cell 10 — Launch chat_web in background + open Colab port tunnel

In [ ]:
import subprocess, time, os, urllib.request

# Kill any previous instance (idempotent re-run)
subprocess.run(["pkill", "-f", "scripts.chat_web"], capture_output=True)
time.sleep(2)

# Start chat_web as a background process
proc = subprocess.Popen(
    ["python", "-m", "scripts.chat_web",
     "-i", "sft", "-g", "d12",
     "--port", "8000", "--host", "127.0.0.1"],
    env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(f"Server PID: {proc.pid}")
print("Waiting for boot (model load + engine warm, ~30–90s)...")

# Poll /health
ready = False
for i in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print(f"✓ Server up after ~{i*2}s")
        ready = True
        break
    except Exception:
        time.sleep(2)
        if proc.poll() is not None:
            print("✗ Server process died. Logs:")
            print(proc.stdout.read() if proc.stdout else "(no output)")
            raise SystemExit

if not ready:
    print("Server didn't come up in 6 min. Recent logs:")
    print(proc.stdout.read() if proc.stdout else "(no output)")
else:
    # Tunnel port 8000 through Colab's proxy
    from google.colab.output import eval_js
    url = eval_js('google.colab.kernel.proxyPort(8000)')
    print(f"\n🔗 Open this URL in a new tab to chat:\n{url}")

## Cell 11 — Stop the web server

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "scripts.chat_web"], capture_output=True)
print("Server stopped.")

---

## Quick reference

| Cell | Purpose | Time | Re-run safe? |
|------|---------|------|-------------|
| 1 | Clone + pip install | ~30s | ✅ |
| 2 | Mount Drive + symlinks | ~5s | ✅ |
| 2b | Install our code into the clone | ~5s | ✅ (run after Cell 1) |
| 3 | GPU sanity check | <1s | ✅ |
| 4 | Download Gutenberg shards | ~10–20 min | ✅ (skips existing) |
| 4b | Stage shards to local SSD | ~1–2 min | ✅ |
| 5 | Train tokenizer | 3–40 min | ✅ (overwrites) |
| 5b | **Pre-tokenize → uint16 shards** | ~2–5 min | ✅ (skips if complete) |
| 6 | Pretrain d12 (`--pretokenized`) | ~110–120 min | ✅ via `--resume-from-step` |
| 7 | Sample from base model | ~30s | ✅ |
| 8 | Download identity data | ~5s | ✅ |
| 9 | SFT | 30–60 min | ⚠️ retrains from scratch each time |
| 10 | Launch web UI + tunnel | ~1 min | ✅ (kills prev instance) |
| 11 | Stop web UI | <1s | ✅ |

## After a session restart

`/content` is wiped but Drive isn't. Re-run cells **1, 2, 2b, 3** (2b re-installs our code into the fresh clone). Cell 4's shards and
the **pre-tokenized data** both live on Drive (via symlink), so if Cell 5b already
completed you can skip Cells 4b/5/5b entirely and jump straight to Cell 6 with
`--resume-from-step=N`. The tokenizer and checkpoints also come back automatically.

## Why pre-tokenization (the whole point of this notebook)

Gutenberg documents are entire **books** (hundreds of KB each). nanochat's default
dataloader tokenizes documents on the fly inside the training step and packs them
with a best-fit buffer; on book-length docs the CPU can't keep up, the GPU starves,
MFU collapses, and the d12 ETA blows past 2h. Pre-tokenizing once into a flat
`uint16` stream removes all per-step CPU work — training just mmaps and slices — so
the GPU runs at full tilt (~56% MFU) and the run finishes in ~110–120 min, the same
ballpark as the `shit_gpt` ClimbMix run.

## Reality check on d12 chat quality

d12 = 286M params on ~1.3B tokens, roughly GPT-1 class. Trained on Gutenberg it will
produce fluent literary prose but weaker factual/world-knowledge than ClimbMix
(which includes web text). After SFT it'll follow user/assistant turns and answer
simple questions, but be confidently wrong often and lose coherence on long
multi-turn. For actual GPT-2 quality, train d24 next (~10x the compute).